In [1]:
import pandas as pd
import numpy as np
import json

file_id = "1xEiGkyyDR0yGoV-C9QeZxqVlWuYfopvd"
url = f"https://drive.google.com/uc?id={file_id}"
df = pd.read_csv(url)

print(f"Shape gốc: {df.shape}")

Shape gốc: (4600, 18)


In [2]:
df_clean = df.copy()

# Trước khi drop
print(f"Trước: {len(df_clean)} rows")

# Drop price lỗi và outlier nặng
df_clean = df_clean[df_clean['price'] > 50_000]
df_clean = df_clean[df_clean['price'] < 2_000_000]

# Drop bedrooms bất thường (0 hoặc > 10)
df_clean = df_clean[(df_clean['bedrooms'] >= 1) & (df_clean['bedrooms'] <= 10)]

# Drop sqft_living bất thường
df_clean = df_clean[df_clean['sqft_living'] > 200]

print(f"Sau:   {len(df_clean)} rows")
print(f"Đã drop: {len(df) - len(df_clean)} rows")

Trước: 4600 rows
Sau:   4499 rows
Đã drop: 101 rows


In [3]:
# Tạo cột mới từ yr_renovated
df_clean['was_renovated'] = (df_clean['yr_renovated'] > 0).astype(int)

# Tuổi nhà tính từ năm bán (dataset này là 2014)
df_clean['house_age'] = 2014 - df_clean['yr_built']

# Log-transform price làm target
df_clean['log_price'] = np.log(df_clean['price'])

# Kiểm tra
print("Các cột mới đã tạo:")
print(df_clean[['price', 'log_price', 'yr_built', 'house_age',
                'yr_renovated', 'was_renovated']].head(5).to_string())

print(f"\nlog_price skewness: {df_clean['log_price'].skew():.4f}")
print(f"price skewness:     {df_clean['price'].skew():.4f}")

Các cột mới đã tạo:
      price  log_price  yr_built  house_age  yr_renovated  was_renovated
0  313000.0  12.653958      1955         59          2005              1
2  342000.0  12.742566      1966         48             0              0
3  420000.0  12.948010      1963         51             0              0
4  550000.0  13.217674      1976         38          1992              1
5  490000.0  13.102161      1938         76          1994              1

log_price skewness: 0.0531
price skewness:     1.6464


In [4]:
# Đây là danh sách feature sẽ dùng để train — thứ tự này quan trọng
# Member B cần biết chính xác danh sách này để build request body

FEATURE_COLS = [
    'sqft_living',    # diện tích sử dụng — correlation cao nhất
    'bedrooms',       # số phòng ngủ
    'bathrooms',      # số phòng tắm
    'floors',         # số tầng
    'waterfront',     # view ven hồ (0/1)
    'view',           # view rating (0-4)
    'condition',      # tình trạng nhà (1-5)
    'sqft_above',     # diện tích tầng trên mặt đất
    'sqft_basement',  # diện tích tầng hầm
    'house_age',      # tuổi nhà
    'was_renovated',  # đã renovate chưa (0/1)
]

TARGET_COL = 'log_price'

# Kiểm tra không có missing trong feature set
print("Missing values trong feature set:")
print(df_clean[FEATURE_COLS + [TARGET_COL]].isnull().sum())

print(f"\nSố features: {len(FEATURE_COLS)}")
print(f"Feature list: {FEATURE_COLS}")

Missing values trong feature set:
sqft_living      0
bedrooms         0
bathrooms        0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
house_age        0
was_renovated    0
log_price        0
dtype: int64

Số features: 11
Feature list: ['sqft_living', 'bedrooms', 'bathrooms', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'house_age', 'was_renovated']


In [5]:
# Chỉ giữ các cột cần thiết
cols_to_save = FEATURE_COLS + [TARGET_COL, 'price', 'city',
                                'lat', 'long', 'date']

# Giữ lại các cột tồn tại trong df_clean
cols_to_save = [c for c in cols_to_save if c in df_clean.columns]

df_export = df_clean[cols_to_save].reset_index(drop=True)

df_export.to_csv('data_clean.csv', index=False)
print(f"Đã export data_clean.csv — {df_export.shape[0]} rows, {df_export.shape[1]} cols")
df_export.head(3)

Đã export data_clean.csv — 4499 rows, 15 cols


,sqft_living,bedrooms,bathrooms,floors,waterfront,view,condition,sqft_above,sqft_basement,house_age,was_renovated,log_price,price,city,date
0,1340,3.0,1.50,1.5,0,0,3,1340,0,59,1,12.653958,313000.0,Shoreline,2014-05-02 00:00:00
1,1930,3.0,2.00,1.0,0,0,4,1930,0,48,0,12.742566,342000.0,Kent,2014-05-02 00:00:00
2,2000,3.0,2.25,1.0,0,0,4,1000,1000,51,0,12.948010,420000.0,Bellevue,2014-05-02 00:00:00


In [6]:
def generate_stats(df):
    stats = {}

    # 1. Summary tổng quan
    stats['summary'] = {
        'total_transactions': int(len(df)),
        'median_price':       int(df['price'].median()),
        'mean_price':         int(df['price'].mean()),
        'max_price':          int(df['price'].max()),
        'min_price':          int(df['price'].min()),
        'date_range':         '2014-05-02 to 2014-07-10'
    }

    # 2. Phân phối giá theo dải
    bins   = [0, 200e3, 300e3, 400e3, 500e3,
              600e3, 700e3, 800e3, 1e6, 2e6]
    labels = ['<200K','200-300K','300-400K','400-500K',
              '500-600K','600-700K','700-800K','800K-1M','1M-2M']
    df['price_bin'] = pd.cut(df['price'], bins=bins, labels=labels)
    dist = df['price_bin'].value_counts().sort_index()
    stats['price_distribution'] = [
        {'range': str(k), 'count': int(v)}
        for k, v in dist.items()
    ]

    # 3. Giá theo thành phố (chỉ city >= 20 records)
    city_stats = (
        df.groupby('city')['price']
        .agg(['median', 'mean', 'count'])
        .query('count >= 20')
        .sort_values('median', ascending=False)
        .reset_index()
    )
    stats['price_by_city'] = [
        {
            'city':   row['city'],
            'median': int(row['median']),
            'mean':   int(row['mean']),
            'count':  int(row['count'])
        }
        for _, row in city_stats.iterrows()
    ]

    # 4. Giá theo dải diện tích
    sqft_bins   = [0, 1000, 1500, 2000, 2500, 3000, 4000, 6000, 15000]
    sqft_labels = ['<1000','1000-1500','1500-2000','2000-2500',
                   '2500-3000','3000-4000','4000-6000','>6000']
    df['sqft_bin'] = pd.cut(df['sqft_living'],
                            bins=sqft_bins, labels=sqft_labels)
    sqft_price = df.groupby('sqft_bin')['price'].median()
    stats['price_by_sqft'] = [
        {'range': str(k), 'median_price': int(v)}
        for k, v in sqft_price.items()
    ]

    # 5. Giá theo số phòng ngủ
    bed_price = (
        df[df['bedrooms'].between(1, 7)]
        .groupby('bedrooms')['price']
        .agg(['median', 'count'])
        .reset_index()
    )
    stats['price_by_bedrooms'] = [
        {
            'bedrooms': int(row['bedrooms']),
            'median':   int(row['median']),
            'count':    int(row['count'])
        }
        for _, row in bed_price.iterrows()
    ]

    # 6. Giá theo view rating
    view_price = (
        df.groupby('view')['price']
        .agg(['median', 'count'])
        .reset_index()
    )
    stats['price_by_view'] = [
        {
            'view':   int(row['view']),
            'median': int(row['median']),
            'count':  int(row['count'])
        }
        for _, row in view_price.iterrows()
    ]

    # 7. Giá theo condition
    cond_price = (
        df.groupby('condition')['price']
        .agg(['median', 'count'])
        .reset_index()
    )
    stats['price_by_condition'] = [
        {
            'condition': int(row['condition']),
            'median':    int(row['median']),
            'count':     int(row['count'])
        }
        for _, row in cond_price.iterrows()
    ]

    # 8. Waterfront premium
    wf = df.groupby('waterfront')['price'].agg(['median', 'count'])
    stats['waterfront_comparison'] = {
        'no_waterfront':  {
            'median': int(wf.loc[0, 'median']),
            'count':  int(wf.loc[0, 'count'])
        },
        'has_waterfront': {
            'median': int(wf.loc[1, 'median']),
            'count':  int(wf.loc[1, 'count'])
        }
    }

    # 9. Giá theo thập niên xây dựng
    df['decade'] = (df['yr_built'] // 10 * 10)
    decade_price = (
        df.groupby('decade')['price']
        .agg(['median', 'count'])
        .reset_index()
    )
    stats['price_by_decade'] = [
        {
            'decade': int(row['decade']),
            'median': int(row['median']),
            'count':  int(row['count'])
        }
        for _, row in decade_price.iterrows()
    ]

    return stats


stats = generate_stats(df_clean.copy())

# Export ra file
with open('stats.json', 'w') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("Đã export stats.json ✓")
print(f"Các keys: {list(stats.keys())}")
print(f"\nSummary: {stats['summary']}")

Đã export stats.json ✓
Các keys: ['summary', 'price_distribution', 'price_by_city', 'price_by_sqft', 'price_by_bedrooms', 'price_by_view', 'price_by_condition', 'waterfront_comparison', 'price_by_decade']

Summary: {'total_transactions': 4499, 'median_price': 460886, 'mean_price': 527537, 'max_price': 1990000, 'min_price': 80000, 'date_range': '2014-05-02 to 2014-07-10'}


/tmp/ipykernel_4749/2791701757.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sqft_price = df.groupby('sqft_bin')['price'].median()


In [7]:
# File này Member B dùng để biết chính xác field nào cần trong request body
feature_config = {
    'feature_cols': FEATURE_COLS,
    'target_col':   TARGET_COL,
    'note': {
        'sqft_living':   'Diện tích sử dụng (sqft)',
        'bedrooms':      'Số phòng ngủ (integer)',
        'bathrooms':     'Số phòng tắm (float, vd: 2.5)',
        'floors':        'Số tầng (float, vd: 1.5)',
        'waterfront':    '0 hoặc 1',
        'view':          '0 đến 4',
        'condition':     '1 đến 5',
        'sqft_above':    'Diện tích trên mặt đất (sqft)',
        'sqft_basement': 'Diện tích tầng hầm (sqft)',
        'house_age':     'Tuổi nhà = 2014 - yr_built',
        'was_renovated': '0 hoặc 1'
    }
}

with open('feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2, ensure_ascii=False)

print("Đã export feature_config.json ✓")
print("\nFeature list theo thứ tự train:")
for i, col in enumerate(FEATURE_COLS):
    print(f"  {i+1:2d}. {col:20s} — {feature_config['note'][col]}")

Đã export feature_config.json ✓

Feature list theo thứ tự train:
   1. sqft_living          — Diện tích sử dụng (sqft)
   2. bedrooms             — Số phòng ngủ (integer)
   3. bathrooms            — Số phòng tắm (float, vd: 2.5)
   4. floors               — Số tầng (float, vd: 1.5)
   5. waterfront           — 0 hoặc 1
   6. view                 — 0 đến 4
   7. condition            — 1 đến 5
   8. sqft_above           — Diện tích trên mặt đất (sqft)
   9. sqft_basement        — Diện tích tầng hầm (sqft)
  10. house_age            — Tuổi nhà = 2014 - yr_built
  11. was_renovated        — 0 hoặc 1


In [8]:
print("=== CHECKLIST TRƯỚC KHI TRAIN ===\n")

df_check = pd.read_csv('data_clean.csv')

print(f"✓ data_clean.csv: {df_check.shape[0]} rows, {df_check.shape[1]} cols")
print(f"✓ Missing values: {df_check[FEATURE_COLS].isnull().sum().sum()} (phải = 0)")
print(f"✓ log_price range: {df_check['log_price'].min():.2f} "
      f"→ {df_check['log_price'].max():.2f}")
print(f"✓ log_price skewness: {df_check['log_price'].skew():.4f} (phải gần 0)")

import os
print(f"\n✓ stats.json:          {'tồn tại' if os.path.exists('stats.json') else 'THIẾU'}")
print(f"✓ feature_config.json: {'tồn tại' if os.path.exists('feature_config.json') else 'THIẾU'}")
print(f"✓ data_clean.csv:      {'tồn tại' if os.path.exists('data_clean.csv') else 'THIẾU'}")

print("\n→ Sẵn sàng qua Notebook 03 — Train model")

=== CHECKLIST TRƯỚC KHI TRAIN ===

✓ data_clean.csv: 4499 rows, 15 cols
✓ Missing values: 0 (phải = 0)
✓ log_price range: 11.29 → 14.50
✓ log_price skewness: 0.0531 (phải gần 0)

✓ stats.json:          tồn tại
✓ feature_config.json: tồn tại
✓ data_clean.csv:      tồn tại

→ Sẵn sàng qua Notebook 03 — Train model
